#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the ephemeral storage, installs ComfyUI, and configures the GGUF integration tools for Qwen-Edit.

import os
import subprocess
from IPython.display import clear_output

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
!pip install -q uv
!cd {LOCAL_WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121
!uv pip install --system -q gguf torchsde websocket-client

GGUF_NODE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-GGUF")
if not os.path.exists(GGUF_NODE_DIR):
    print("🧩 [@CoinNoin] Installing GGUF Processing Nodes...")
    !git clone https://github.com/city96/ComfyUI-GGUF {GGUF_NODE_DIR} &> /dev/null
    !uv pip install --system -q -r {GGUF_NODE_DIR}/requirements.txt
else:
    !cd {GGUF_NODE_DIR} && git pull &> /dev/null

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader & LoRA Setup
#@markdown The **Qwen-Image-Edit Q3_K_M GGUF**, FP8 Text Encoders, VAE, and **Lightning LoRA** are fetched automatically.

# --- Optional Custom LoRA Source ---
LORA_SOURCE = "None (Use Built-in Lightning LoRA)" #@param ["None (Use Built-in Lightning LoRA)", "Download from URL", "Upload from Computer"]
LORA_URLS = "" #@param {type:"string"}

import os
import subprocess
import urllib.parse
from google.colab import files

WORKSPACE = "/content/ComfyUI"

# Fixed Models for Qwen-Image-Edit
FIXED_UNET_URL = "https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q3_K_M.gguf"
TEXT_ENCODER_URL = "https://huggingface.co/Comfy-Org/HunyuanVideo_1.5_repackaged/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors"
VAE_URL = "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors"
LIGHTNING_LORA_URL = "https://huggingface.co/lightx2v/Qwen-Image-Edit-2511-Lightning/resolve/main/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"

DIRS = {
    "unet":  os.path.join(WORKSPACE, "models/unet"),
    "clip":  os.path.join(WORKSPACE, "models/clip"),
    "text_encoders": os.path.join(WORKSPACE, "models/text_encoders"),
    "vae":   os.path.join(WORKSPACE, "models/vae"),
    "loras": os.path.join(WORKSPACE, "models/loras"),
    "input": os.path.join(WORKSPACE, "input"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("⚡ [@CoinNoin] Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download(url, target_dir, filename=None):
    if not url.strip(): return
    if not filename:
        parsed_url = urllib.parse.urlparse(url)
        filename = os.path.basename(parsed_url.path)
    if os.path.exists(os.path.join(target_dir, filename)) and os.path.getsize(os.path.join(target_dir, filename)) > 1000000:
        print(f"   ✅ Already Cached: {filename}")
        return

    print(f"   📥 Fetching: {filename}...")
    cmd = [
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ]
    subprocess.run(cmd, check=True)
    print(f"   ✅ Saved in {os.path.basename(target_dir)}")

# Download required base assets
print("\n📥 [@CoinNoin] Commencing Model Downloads...")
download(FIXED_UNET_URL, DIRS["unet"])
download(TEXT_ENCODER_URL, DIRS["clip"])
download(VAE_URL, DIRS["vae"])
download(LIGHTNING_LORA_URL, DIRS["loras"])

# Symlink text encoder for compatibility
clip_src = os.path.join(DIRS["clip"], "qwen_2.5_vl_7b_fp8_scaled.safetensors")
clip_dst = os.path.join(DIRS["text_encoders"], "qwen_2.5_vl_7b_fp8_scaled.safetensors")
if os.path.exists(clip_src) and not os.path.exists(clip_dst):
    os.symlink(clip_src, clip_dst)

# Handle optional custom LoRA
ACTIVE_LORA_FILENAME = ""
if LORA_SOURCE == "Download from URL":
    if LORA_URLS.strip():
        parsed_name = os.path.basename(urllib.parse.urlparse(LORA_URLS.strip()).path)
        if not parsed_name.endswith(('.safetensors', '.pt', '.ckpt')):
            parsed_name = "custom_downloaded_lora.safetensors"
        download(LORA_URLS.strip(), DIRS["loras"], filename=parsed_name)
        ACTIVE_LORA_FILENAME = parsed_name
    else:
        print("⚠️ No custom LoRA URL provided. Proceeding with defaults.")
elif LORA_SOURCE == "Upload from Computer":
    print("\n📤 [@CoinNoin] Select your custom LoRA (.safetensors) file to upload:")
    uploaded = files.upload()
    if uploaded:
        for fname in uploaded.keys():
            dest = os.path.join(DIRS["loras"], fname)
            os.replace(fname, dest)
            ACTIVE_LORA_FILENAME = fname
            print(f"   ✅ Custom LoRA uploaded: {fname}")
    else:
        print("⚠️ No file uploaded. Proceeding with defaults.")

available_loras = [f for f in os.listdir(DIRS["loras"]) if f.endswith(('.safetensors', '.pt', '.ckpt'))]
print("\n📂 [@CoinNoin] Available LoRAs in storage:")
for idx, lora_name in enumerate(available_loras, 1):
    print(f"   {idx}. {lora_name}")

print("\n✅ [@CoinNoin] All core assets secured!")

In [ ]:
#@title 3. Qwen-Image-Edit Generation
#@markdown Upload your target image (and optional reference). The output automatically matches Image 1's native dimensions.

# --- Upload Controls ---
UPLOAD_IMAGE_1 = True #@param {type:"boolean"}
#@markdown *Image 1 is REQUIRED. This is the image you want to edit.*
UPLOAD_IMAGE_2 = False #@param {type:"boolean"}
#@markdown *Image 2 is OPTIONAL. Use this as a visual reference (e.g., clothes).*

# --- Edit Instructions ---
PROMPT = "Change the clothes of the person in image 1 to match the outfit in image 2. Keep the face, pose, and body identical." #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, artifacts, double clothes, layering, overlapping outfits" #@param {type:"string"}

# --- LoRA Selection ---
USE_LIGHTNING_LORA = True #@param {type:"boolean"}
#@markdown *Recommended for T4: Forces Steps=4 and CFG=1.0 for ~15s generations.*
CUSTOM_LORA = "None" #@param ["None", "auto"] {allow-input: true}
#@markdown *If Lightning is OFF, type the exact custom LoRA filename to use it.*

# --- Sampling Settings ---
STEPS = 4 #@param {type:"slider", min:1, max:50, step:1}
CFG = 1.0 #@param {type:"slider", min:1.0, max:8.0, step:0.1}
SHIFT = 3.0 #@param {type:"slider", min:1.0, max:10.0, step:0.1}
SEED = 0 #@param {type:"integer"}

# --- Direct Download ---
AUTO_DOWNLOAD = False #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from PIL import Image
from google.colab import files
from IPython.display import display, Image as IPImage, clear_output, HTML

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
os.makedirs(INPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

# Fixed Models for Qwen-Image-Edit
GGUF_UNET = "qwen-image-edit-2511-Q3_K_M.gguf"
TEXT_ENCODER = "qwen_2.5_vl_7b_fp8_scaled.safetensors"
VAE_NAME = "qwen_image_vae.safetensors"
TURBO_LORA = "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"

# Auto-adjust steps/cfg for Lightning
if USE_LIGHTNING_LORA:
    STEPS = min(STEPS, 4)
    CFG = 1.0

if SEED == 0:
    SEED = random.randint(1, 2**31)

print("🔌 [@CoinNoin] Checking ComfyUI Server Status...")
def start_server():
    req = urllib.request.Request("http://127.0.0.1:8188")
    try:
        urllib.request.urlopen(req)
        print("   🟢 Server is already running.")
    except:
        print("   🚀 Starting ComfyUI Server in background...")
        subprocess.Popen([sys.executable, "main.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        while True:
            try:
                urllib.request.urlopen(req)
                print("   🟢 Server is now up and ready!")
                break
            except:
                time.sleep(2)

start_server()
clear_output()

# --- Handle Uploads & Thumbnails ---
img1_name = "qwen_edit_img1.png"
img2_name = "qwen_edit_img2.png"
has_img2 = False

if UPLOAD_IMAGE_1:
    print("📤 [@CoinNoin] Please upload IMAGE 1 (The image to edit):")
    uploaded = files.upload()
    if uploaded:
        source_name = list(uploaded.keys())[0]
        os.replace(source_name, os.path.join(INPUT_DIR, img1_name))
        print("   ✅ Image 1 saved!")
    else:
        print("   ❌ No image uploaded. Will attempt to use cached image.")

input_path = os.path.join(INPUT_DIR, img1_name)
if not os.path.exists(input_path):
    raise RuntimeError("❌ No input Image 1 found! Please check 'UPLOAD_IMAGE_1' and run again.")

print("\n🖼️ [@CoinNoin] Image 1 Preview:")
display(IPImage(filename=input_path, height=256))

if UPLOAD_IMAGE_2:
    print("\n📤 [@CoinNoin] Please upload IMAGE 2 (The visual reference):")
    uploaded_sec = files.upload()
    if uploaded_sec:
        sec_name = list(uploaded_sec.keys())[0]
        os.replace(sec_name, os.path.join(INPUT_DIR, img2_name))
        has_img2 = True
        print("   ✅ Image 2 saved!")
elif os.path.exists(os.path.join(INPUT_DIR, img2_name)):
    has_img2 = True

if has_img2:
    print("\n🖼️ [@CoinNoin] Image 2 Preview:")
    display(IPImage(filename=os.path.join(INPUT_DIR, img2_name), height=256))

# Brand visibility prefix in filename
BRAND_PREFIX = "@CoinNoin_QwenEdit"

print(f"\n\033[94m➜ [@CoinNoin] Edit Details | Steps: {STEPS} | CFG: {CFG} | Aura Shift: {SHIFT} | Seed: {SEED}\033[0m")
print(f"📝 Prompt: '{PROMPT}'")

# --- Build Qwen-Image-Edit Architecture ---
workflow = {
    "unet": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": GGUF_UNET}},
    "clip": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENCODER, "type": "qwen_image", "device": "default"}},
    "vae":  {"class_type": "VAELoader", "inputs": {"vae_name": VAE_NAME}},

    # Load Image 1 at native resolution
    "load1": {"class_type": "LoadImage", "inputs": {"image": img1_name}},

    # Encode to latent (uses Image 1's native dimensions)
    "vae_enc": {"class_type": "VAEEncode", "inputs": {"pixels": ["load1", 0], "vae": ["vae", 0]}},

    # AuraFlow + CFGNorm
    "aura": {"class_type": "ModelSamplingAuraFlow", "inputs": {"shift": SHIFT, "model": ["unet", 0]}},
    "cfgnorm": {"class_type": "CFGNorm", "inputs": {"strength": 1.0, "model": ["aura", 0]}},

    # Conditioning (uses Image 1 + optional Image 2)
    "pos": {"class_type": "TextEncodeQwenImageEditPlus", "inputs": {
        "clip": ["clip", 0], "vae": ["vae", 0], "image1": ["load1", 0], "prompt": PROMPT}},
    "neg": {"class_type": "TextEncodeQwenImageEditPlus", "inputs": {
        "clip": ["clip", 0], "vae": ["vae", 0], "image1": ["load1", 0], "prompt": NEGATIVE_PROMPT}},

    # KSampler (latent inherits Image 1's size)
    "sampler": {"class_type": "KSampler", "inputs": {
        "seed": SEED, "steps": STEPS, "cfg": CFG,
        "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
        "model": ["cfgnorm", 0],
        "positive": ["pos", 0], "negative": ["neg", 0],
        "latent_image": ["vae_enc", 0]}},

    # Decode + Save
    "decode": {"class_type": "VAEDecode", "inputs": {"samples": ["sampler", 0], "vae": ["vae", 0]}},
    "save": {"class_type": "SaveImage", "inputs": {"filename_prefix": BRAND_PREFIX, "images": ["decode", 0]}},
}

# Inject Image 2 if present
if has_img2:
    workflow["load2"] = {"class_type": "LoadImage", "inputs": {"image": img2_name}}
    workflow["pos"]["inputs"]["image2"] = ["load2", 0]
    workflow["neg"]["inputs"]["image2"] = ["load2", 0]

# Inject LoRA
lora_to_use = None
if USE_LIGHTNING_LORA:
    lora_to_use = TURBO_LORA
    print("⚡ [@CoinNoin] Using Lightning 4-Step LoRA")
elif CUSTOM_LORA.strip() and CUSTOM_LORA.lower() != "none":
    if CUSTOM_LORA.lower() == "auto" and 'ACTIVE_LORA_FILENAME' in globals() and ACTIVE_LORA_FILENAME:
        lora_to_use = ACTIVE_LORA_FILENAME
    else:
        lora_to_use = CUSTOM_LORA
    print(f"✨ [@CoinNoin] Using Custom LoRA: {lora_to_use}")

if lora_to_use:
    workflow["lora"] = {"class_type": "LoraLoaderModelOnly", "inputs": {
        "lora_name": lora_to_use, "strength_model": 1.0, "model": ["aura", 0]}}
    workflow["cfgnorm"]["inputs"]["model"] = ["lora", 0]

p = {"prompt": workflow}
data = json.dumps(p).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)

print("\n📥 Submitting workflow to API...")
try:
    response = urllib.request.urlopen(req)
    prompt_id = json.loads(response.read())['prompt_id']
except Exception as e:
    print(f"❌ API Error: {e}")
    raise

print("✨ Processing and Generating (This may take 30-120s on T4)...")
while True:
    try:
        history_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
        history_res = urllib.request.urlopen(history_req)
        history_data = json.loads(history_res.read())
        if prompt_id in history_data:
            outputs = history_data[prompt_id]['outputs']
            break
    except:
        pass
    time.sleep(1)

print("\n🖼️ Decoding Final Masterpiece:")
generated_images = []
for node_id, node_output in outputs.items():
    if 'images' in node_output:
        for image in node_output['images']:
            filename = image['filename']
            img_path = os.path.join(WORKSPACE, "output", filename)
            generated_images.append(img_path)
            print(f"\033[92m✓ [@CoinNoin] Saved: {filename}\033[0m")
            display(IPImage(filename=img_path))

# Direct download trigger
if AUTO_DOWNLOAD and generated_images:
    print("\n📥 [@CoinNoin] Download Started...")
    for img_file in generated_images:
        files.download(img_file)